In [0]:
ruta = "/Volumes/electrocasa_dev/bronze/landing/ventas"

display(dbutils.fs.ls(ruta))

In [0]:
schema_ventas = "/Volumes/electrocasa_dev/bronze/landing/schemas/ventas"

ventas = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", schema_ventas)
        .option("rescuedDataColumn", "_rescued_data")
        .load(ruta)
)

ventas.printSchema()

In [0]:
%sql

SELECT
COUNT(*) AS total_registros,
COUNT(DISTINCT venta_id) AS ventas_unicas,
COUNT(DISTINCT archivo_origen) AS archivos_origen,
SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS registros_rescatados
FROM electrocasa_dev.bronze.ventas;

In [0]:
%sql

SELECT
venta_id,
fec_ingesta,
archivo_origen,
id_lote,
_rescued_data
FROM electrocasa_dev.bronze.ventas
LIMIT 10;

In [0]:
%sql

SELECT
COUNT(*) AS ids_repetidos,
SUM(cantidad - 1) AS filas_adicionales,
MAX(cantidad) AS max_repeticiones
FROM (
    SELECT
        venta_id,
        COUNT(*) AS cantidad
    FROM electrocasa_dev.bronze.ventas
    GROUP BY venta_id
    HAVING COUNT(*) > 1
);

In [0]:
%sql

SELECT *
FROM electrocasa_dev.bronze.ventas
WHERE venta_id IN (
SELECT venta_id
FROM electrocasa_dev.bronze.ventas
GROUP BY venta_id
HAVING COUNT(*) > 1
LIMIT 5
)
ORDER BY venta_id;

In [0]:
%sql

SELECT
COUNT(*) AS total_registros,
SUM(CASE
        WHEN monto_total IS NULL OR monto_total = ''
        THEN 1 ELSE 0
    END) AS monto_nulo,
SUM(CASE
        WHEN TRY_CAST(monto_total AS DOUBLE) <= 0
        THEN 1 ELSE 0
    END) AS monto_no_positivo,
SUM(CASE
        WHEN TRY_CAST(cantidad AS INT) <= 0
        THEN 1 ELSE 0
    END) AS cantidad_no_positiva,
SUM(CASE
        WHEN sucursal_id IS NULL OR TRIM(sucursal_id) = ''
        THEN 1 ELSE 0
    END) AS sucursal_faltante
FROM electrocasa_dev.bronze.ventas;

In [0]:
%sql

SELECT
    CASE
        WHEN fecha_venta RLIKE '^\\d{4}-\\d{2}-\\d{2}$'
            THEN 'yyyy-MM-dd'
        WHEN fecha_venta RLIKE '^\\d{2}/\\d{2}/\\d{4}$'
            THEN 'dd/MM/yyyy'
        ELSE 'otro'
    END AS formato_fecha,
    COUNT(*) AS cantidad
FROM electrocasa_dev.bronze.ventas
GROUP BY
    CASE
        WHEN fecha_venta RLIKE '^\\d{4}-\\d{2}-\\d{2}$'
            THEN 'yyyy-MM-dd'
        WHEN fecha_venta RLIKE '^\\d{2}/\\d{2}/\\d{4}$'
            THEN 'dd/MM/yyyy'
        ELSE 'otro'
    END
ORDER BY cantidad DESC;

In [0]:
%sql

SELECT
metodo_pago,
COUNT(*) AS cantidad
FROM electrocasa_dev.bronze.ventas
GROUP BY metodo_pago
ORDER BY cantidad DESC;

In [0]:
%sql

SELECT
COUNT(*) AS total_registros,
COUNT(DISTINCT venta_id) AS ventas_unicas
FROM electrocasa_dev.silver.ventas;

In [0]:
%sql

DESCRIBE TABLE electrocasa_dev.silver.ventas;

In [0]:
%sql

SELECT
metodo_pago,
COUNT(*) AS cantidad
FROM electrocasa_dev.silver.ventas
GROUP BY metodo_pago
ORDER BY cantidad DESC;

In [0]:
%sql

SELECT
COUNT(*) AS total_registros,
SUM(CASE
        WHEN monto_total IS NULL
        THEN 1 ELSE 0
    END) AS monto_nulo,
SUM(CASE
        WHEN monto_total <= 0
        THEN 1 ELSE 0
    END) AS monto_no_positivo,
SUM(CASE
        WHEN monto_total IS NULL OR monto_total <= 0
        THEN 1 ELSE 0
    END) AS filas_invalidas_monto
FROM electrocasa_dev.silver.ventas;